# expiMap reproduction
Runs 4 variants (baseline, extra layer, correlation penalty, both) x 3 seeds, then writes `results/results.csv` and `results/results_summary.csv`.

In [2]:
!nvidia-smi | head -n 12
import torch
print(torch.__version__, torch.cuda.is_available())

Mon Nov  2 17:25:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
REPO = 'https://github.com/Hastikabirsamadi/Bioinformatics-course-project-scArches-.git'
%cd /content
!git clone $REPO repo || git -C repo pull
%cd repo

/content
Cloning into 'repo'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 123 (delta 21), reused 120 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 120.15 KiB | 1008.00 KiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/repo


In [4]:
%pip install -q -r requirements-colab.txt
!python download_data.py

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
Downloading...
From (original): https://drive.google.com/uc?id=1Rnm-XKEqPLdOq3lpa3ka2aV4bOXVCLP0
From (redirected): https://drive.google.com/uc?id=1Rnm-XKEqPLdOq3lpa3ka2aV4bOXVCLP0&confirm=t&uuid=0e3cd177-63f2-4ed2-9560-4180f581c384
To: /content/repo/data/pbmc_tutorial.h5ad
100% 231M/231M [00:03<00:00, 73.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1t3oMuUfueUz_caLm5jmaEYjBxVNSsfxG
To:

In [5]:
# quick sanity run
!OUT=results_smoke/runs SEEDS=0 VARIANTS='baseline both' PY=python ./run_all.sh --epochs 3 --max_cells 3000

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
I0000 00:00:1793640400.997200    1162 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793640403.954723    1162 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
 captum (see https://github.com/pytorch/captum).
variant=baseline seed=0 device=cuda

INITIALIZING NEW NETWORK..............
Encoder Architecture:
	Input Layer in, out and cond: 1972 256 4
	Hidden Layer 1 in/out: 256 256
	Hidden Layer 2 in/out: 256 256
	Mean/Var Layer in/out: 256 282
Decoder Architecture:
	Masked linear layer in, ext_m, ext, cond, out:  282 0 0

In [6]:
# connect Google Drive: results are saved there, so a disconnect cannot erase finished runs
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# full run: 4 variants x 3 seeds, written straight to Google Drive.
# If the session ends, reconnect, run the clone, install and Drive cells, then run this cell again.
# Finished runs are skipped.
!OUT=/content/drive/MyDrive/expimap_results/runs PY=python ./run_all.sh --save_models

skip baseline seed0 (done)
skip baseline seed1 (done)
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
I0000 00:00:1793640617.305974    2498 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793640620.862403    2498 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
 captum (see https://github.com/pytorch/captum).
variant=baseline seed=2 device=cuda

INITIALIZING NEW NETWORK..............
Encoder Architecture:
	Input Layer in, out and cond: 1972 256 4
	Hidden Layer 1 in/out: 256 256
	Hidden Layer 2 in/out: 256 256
	Mean/Var Layer in/out: 256 282
Decoder Architecture:
	M

In [ ]:
import pandas as pd
pd.read_csv('/content/drive/MyDrive/expimap_results/results_summary.csv', index_col=0).T

In [ ]:
# download a zip of the results (the models are left out to keep it small)
from google.colab import files
!cd /content/drive/MyDrive && zip -qr /content/results.zip expimap_results -x 'expimap_results/runs/*/*/ref_model/*' 'expimap_results/runs/*/*/query_model/*'
files.download('/content/results.zip')